# VERA · MimicGen **Stack** (client–server, minimalist)

Drives the MimicGen `stack_d0` two-block stacking task against a running VERA policy server.
Inference runs on the server's GPU; this notebook only steps the env.

**Start the server first** (Terminal 1) — point at the downloaded checkpoints, then launch:
```bash
export VERA_WAN_CKPT_ROOT=/path/to/Wan2.1-T2V-1.3B            # frozen Wan2.1 base (text-enc + VAE)
export VERA_MIMICGEN_CKPT_DIR=./vera-ckpts/mimicgen-wan-1.3b  # specialist DiT + flow decoder
python -m vera.server.start_vera_server --embodiment mimicgen --port 8800 --vis-port 8801 \
    --algo-config $VERA_MIMICGEN_CKPT_DIR/algo_config.yaml \
    --text "A robot arm stacks one block on top of another block"
```
The Jacobian IDM checkpoint loads locally via `VERA_MIMICGEN_DYNAMICS_CKPT`
(default: `./vera-ckpts/idm-mimicgen-285ouq1q/model.ckpt`).
Watch the **live viewer at http://localhost:8801/** (dream · tracks · jacobian · per-chunk play bars) while rollouts run.

In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')
import numpy as np
import vera
print('vera:', vera.__file__)

vera: /path/to/VERA/vera/__init__.py


In [2]:
HOST, PORT, VIS_PORT = "127.0.0.1", 8800, 8801
DATASET = "/path/to/mimicgen_datasets/core/stack_d0.hdf5"

# Walkthrough default: one demo. Set DEMO_KEYS = None to evaluate across demos (population SR).
DEMO_KEYS = ["demo_80"]
NUM_DEMOS = 1
ROLLOUT_HORIZON = 700
RENDER_SIZE = 128
DEMO_INDICES = None
PROMPT = "A robot arm stacks one block on top of another block"

In [3]:
# --- Connect: attach to the running server + print what it's serving ---
from vera.server.protocol.websocket_policy_client import WebsocketClientPolicy
client = WebsocketClientPolicy(host=HOST, port=PORT)
meta = client.get_server_metadata()
view_keys = list(meta['view_keys'])
context_frames = int(meta.get('context_frames', 9))
print('planner (WAN):', meta.get('planner_model'))
print('IDM         :', meta.get('idm_model'))
print('views       :', view_keys, '| context_frames:', context_frames)
print('action_space:', meta.get('action_space'), '| H:', meta.get('action_horizon'), '| control_hz:', meta.get('control_hz'))
print(f'live viewer : http://localhost:{VIS_PORT}/')

planner (WAN): ./vera-ckpts/mimicgen-wan-1.3b/algo_config.yaml
IDM         : idm-mimicgen-285ouq1q
views       : ['agentview_image', 'robot0_eye_in_hand_image'] | context_frames: 21
action_space: eef_delta | H: 10 | control_hz: None
live viewer : http://localhost:8801/


In [4]:
# --- Build the MimicGen runner + a RemotePolicy that forwards inference to the server ---
from vera.env_runner.mimicgen_runner import MimicgenRunner, MimicgenRunnerCfg
from vera.controller.run_mimicgen_eval import RemotePolicy
import torch

cfg = MimicgenRunnerCfg(
    env_name='mimicgen', dataset_path=DATASET, render_size=RENDER_SIZE,
    render_obs_key=view_keys, num_demos_to_run=NUM_DEMOS,
    max_episode_steps=ROLLOUT_HORIZON, n_repeat=1, action_scale=1.0,
    save_videos=True, save_trajectory=False, save_rrd=False,
    output_dir='outputs/vera_mimicgen_stack', use_stored_model=False,
    demo_warmup_steps=max(context_frames - 1, 0), log_step_debug=False,
)
runner = MimicgenRunner(cfg, device=torch.device('cpu'))
runner.setup_env()
remote = RemotePolicy(client, view_keys=view_keys,
                      view_widths=[RENDER_SIZE] * len(view_keys), context_frames=context_frames,
                      prompt=PROMPT)
print('runner + remote policy ready')

/path/to/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


ROBOMIMIC WARNING(
    No private macro file found!
    It is recommended to use a private macro file
    To setup, run: python /path/to/repo/third_party/robomimic/robomimic/scripts/setup_macros.py
)


[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /path/to/repo/third_party/robosuite/robosuite/scripts/setup_macros.py (macros.py:55)


Got error: No module named 'robosuite_task_zoo'
[MimicgenRunner] Overriding image size to 128x128 (dataset was 84x84)

============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: []
using obs modality: rgb with keys: ['robot0_eye_in_hand_image', 'agentview_image']
[MimicgenRunner] Overriding image size to 128x128 (dataset was 84x84)

============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: []
using obs modality: rgb with keys: ['robot0_eye_in_hand_image', 'agentview_image']
runner + remote policy ready


In [5]:
if DEMO_KEYS:
    options = {'demo_keys': list(DEMO_KEYS)}
elif DEMO_INDICES is not None:
    options = {'demo_indices': DEMO_INDICES}
else:
    options = None
results = runner.run(remote, run_tag='vera_stack', options=options)

succ    = np.asarray(results.get('env_successes', []), dtype=bool)
relaxed = np.asarray(results.get('relaxed_successes', succ), dtype=bool)
maxr    = np.asarray(results.get('max_rewards', []), dtype=float)
n = len(succ)
print(f'success:  {int(succ.sum())}/{n}' + (f' = {100*succ.mean():.0f}%' if n else ''))
print(f'relaxed:  {int(relaxed.sum())}/{n}')
print(f'maxR mean: {maxr.mean() if maxr.size else 0:.3f}   (sparse: >0 only on a completed stack)')
print('videos:', results.get('save_dir'))

/path/to/site-packages/gymnasium/utils/passive_env_checker.py:159: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
[MimicgenRunner] Rollout:   0%|          | 0/700 [00:00<?, ?it/s]

    chunk   1 (env step   1): dream done in  9.4s → committing 10 actions (|a|=0.255)            


/path/to/site-packages/gymnasium/utils/passive_env_checker.py:159: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
[MimicgenRunner] Rollout:   1%|▏         | 9/700 [00:09<08:54,  1.29it/s]  

    chunk   2 (env step  11): dreaming + denoising on server…

[MimicgenRunner] Rollout:   1%|▏         | 9/700 [00:19<08:54,  1.29it/s]

...

[MimicgenRunner] Rollout:  27%|██▋       | 187/700 [03:18<08:43,  1.02s/it]

    chunk  20 (env step 191): dream done in 10.6s → committing 10 actions (|a|=0.170)            


[MimicgenRunner] Rollout:  27%|██▋       | 192/700 [03:29<09:14,  1.09s/it]


[MimicgenRunner] Rollout complete.
  Demo returns: [11.]  (mean=11.000)


success:  1/1 = 100%
relaxed:  1/1
maxR mean: 1.000   (sparse: >0 only on a completed stack)
videos: outputs/vera_mimicgen_stack/run_20260623_012640_vera_stack


In [6]:
from vera.env_runner.mimicgen_runner import format_run_results
r = format_run_results(results)
r.show(fps=5, height=252)


─── Metrics ───
  train_returns: [11.]
  eval_returns: []
  max_rewards: [1.]
  max_reward_mean: 1.0
  save_dir: outputs/vera_mimicgen_stack/run_20260623_012640_vera_stack

─── obs ───


This browser does not support the video tag.



─── policy ───


This browser does not support the video tag.



─── agentview_image ───


This browser does not support the video tag.



─── robot0_eye_in_hand_image ───


This browser does not support the video tag.


In [ ]:
import urllib.request, json, time, base64, tempfile
import cv2, numpy as np, mediapy as media
from pathlib import Path
from IPython.display import HTML, display

def show_policy_vis(vis_host=HOST, vis_port=VIS_PORT, fps=10, max_width=1100):
    base = f"http://{vis_host}:{vis_port}"
    try:
        with urllib.request.urlopen(f"{base}/stats.json", timeout=5) as r:
            n = int(json.loads(r.read().decode()).get("video_len", 0))
    except Exception as e:
        print(f"[vis] server not reachable at {base} - is it up with --vis-port {vis_port}? ({e})"); return
    if n == 0:
        print("[vis] buffer empty - run the rollout cell first (watch the live viewer while it runs)."); return
    urllib.request.urlopen(f"{base}/video/pause", timeout=3)
    frames = []
    for i in range(n):
        urllib.request.urlopen(f"{base}/video/seek?pos={i}", timeout=3); time.sleep(0.05)
        with urllib.request.urlopen(f"{base}/policy.mjpg", timeout=5) as resp:
            buf = b""
            while True:
                chunk = resp.read(4096)
                if not chunk: break
                buf += chunk
                e = buf.find(b"\xff\xd9")
                if e != -1:
                    s = buf.find(b"\xff\xd8")
                    if s != -1:
                        img = cv2.imdecode(np.frombuffer(buf[s:e+2], np.uint8), cv2.IMREAD_COLOR)
                        if img is not None: frames.append(img)
                    break
    urllib.request.urlopen(f"{base}/video/play", timeout=3)
    if not frames:
        print("[vis] no frames grabbed."); return
    Hm = max(f.shape[0] for f in frames); Wm = max(f.shape[1] for f in frames)
    def _pad(f):
        h, w = f.shape[:2]
        if (h, w) == (Hm, Wm): return f
        out = np.zeros((Hm, Wm, 3), dtype=f.dtype); out[:h, :w] = f; return out
    rgb = np.stack([cv2.cvtColor(_pad(f), cv2.COLOR_BGR2RGB) for f in frames])
    mp4 = tempfile.mktemp(suffix=".mp4")
    media.write_video(mp4, rgb, fps=fps)
    b64 = base64.b64encode(Path(mp4).read_bytes()).decode()
    print(f"[vis] composite policy-vis: {len(rgb)} frames @ {Wm}x{Hm}")
    display(HTML(
        "<p><b>[ current | dream + tracks &#8594; dream | jacobian ]</b></p>"
        f'<video width="{min(Wm, max_width)}" controls loop autoplay muted>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

show_policy_vis()
